# CellLineSelector — Exploratory Data Analysis

Reads `celllineselector.db` (built by the harmonisation pipeline) and profiles it
along four axes: **completeness**, **scale**, **shape**, and **agreement**.

| § | question |
|---|---|
| 2 | What's in the database? Shapes, keys, identity ambiguity |
| 3 | How complete is it? Missingness, per-layer and per-line coverage |
| 4 | What are the cell lines? Disease, lineage, demographics |
| 5 | Are the layers on comparable scales? Value ranges, skewness, distributions |
| 6 | What genes are covered, and by which sources? |
| 7 | Do independent sources agree? Same-axis corroboration |
| 8 | Single-gene retrieval and its validation |

Read-only throughout — nothing here modifies the database.

In [ ]:
import re
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from itertools import combinations
from scipy.stats import gaussian_kde

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})

import os
# built by 00_harmonisation.ipynb (+ 00b for the enriched tables)
DB_PATH = os.path.join(os.path.abspath(os.path.join(os.getcwd(), "outputs")),
                       "celllineselector.db")
con = duckdb.connect(DB_PATH, read_only=True)

# Layer groupings used throughout. MODALITY = measured evidence;
# ANNOTATION = identity/metadata, which describes a line rather than assaying it.
MODALITY_LAYERS   = ["depmap_expr", "geo_expr", "hpa_rna", "mutations", "fusions",
                     "proteomics", "metabolomics", "mirna", "signatures"]
ANNOTATION_LAYERS = ["sample_info", "cellosaurus", "hpa_desc", "depmap_profiles", "geo_info"]
ALL_LAYERS        = MODALITY_LAYERS + ANNOTATION_LAYERS

# Layers whose VALUES are numeric measurements (excludes pure event/annotation tables)
NUMERIC_LAYERS = ["depmap_expr", "geo_expr", "hpa_rna", "proteomics",
                  "metabolomics", "mirna", "signatures"]

# Numeric-typed columns that are identifiers/counters, not measurements
SKIP_COLS = {"model_id", "sample", "profileid", "ccle_name", "ccle_id", "gene", "name",
             "n_model_id", "is_ambiguous", "model_id_unresolved", "age", "n_geo",
             "n_rrid", "n_rrids", "n_profile_id", "pos", "start", "end"}

ENSG_PATTERN = r'^ensg\d+'

print(con.execute("SHOW TABLES").df().to_string(index=False))

## 1. Shared helpers

One definition each — the original notebook defined `numeric_cols` and
`validate_gene_fetch` twice with different bodies, and opened nine separate
connections.

In [ ]:
def table_exists(con, t):
    return con.execute("SELECT count(*) FROM information_schema.tables "
                       "WHERE lower(table_name)=?", [t.lower()]).fetchone()[0] > 0


def cols_of(con, t):
    return [r[0] for r in con.execute(f'DESCRIBE "{t}"').fetchall()]


def has_col(con, t, c):
    return table_exists(con, t) and c in cols_of(con, t)


def numeric_cols(con, table, skip=SKIP_COLS):
    """Numeric columns that represent measurements, not identifiers."""
    d = con.execute(f'DESCRIBE "{table}"').df()
    num = d[d["column_type"].str.upper().str.contains(
        "INT|DOUBLE|FLOAT|DECIMAL|REAL|BIGINT|HUGEINT", regex=True)]
    return [c for c in num["column_name"] if c.lower() not in skip]


def wide_genes(con, table, prefix="ensg"):
    """Gene columns of a wide expression table."""
    return [c for c in cols_of(con, table) if c.lower().startswith(prefix)]


def chunked_agg(con, table, cols, expr_fn, chunk=300):
    """
    Apply a per-column aggregate across many columns, in chunks.
    A 54,000-term SELECT exceeds DuckDB's expression limit; chunking keeps it
    to a handful of fast queries. Returns a Series indexed by column name.
    """
    parts = []
    for i in range(0, len(cols), chunk):
        ch = cols[i:i + chunk]
        exprs = ", ".join(f'{expr_fn(c)} AS "{c}"' for c in ch)
        parts.append(con.execute(f'SELECT {exprs} FROM "{table}"').df().iloc[0])
    return pd.concat(parts).astype(float) if parts else pd.Series(dtype=float)


def style_axis(ax, grid_axis="x"):
    ax.grid(axis=grid_axis, color="#EDEDED", lw=.7, zorder=0)
    ax.set_axisbelow(True)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    for sp in ("left", "bottom"):
        ax.spines[sp].set_color("#CCC")
    ax.tick_params(length=0, labelsize=9)

## 2. What's in the database?

### 2.1 Table shapes and keys

In [ ]:
def table_summary(con):
    rows = []
    for t in con.execute("SHOW TABLES").df()["name"]:
        n_rows = con.execute(f'SELECT count(*) FROM "{t}"').fetchone()[0]
        cs = cols_of(con, t)
        n_ids = (con.execute(f'SELECT count(DISTINCT model_id) FROM "{t}"').fetchone()[0]
                 if "model_id" in cs else None)
        rows.append({"table": t, "rows": n_rows, "cols": len(cs),
                     "distinct_model_id": n_ids,
                     "gene_cols": len([c for c in cs if c.lower().startswith("ensg")])})
    return pd.DataFrame(rows).sort_values("rows", ascending=False)


summary = table_summary(con)
print(summary.to_string(index=False))

### 2.2 Identity ambiguity

Rows whose CVCL or GSM resolved to more than one candidate ACH were duplicated
during harmonisation ("credit both") and flagged with `is_ambiguous`. This is
the audit of how much duplication that introduced.

In [ ]:
amb = []
for t in ALL_LAYERS:
    if not has_col(con, t, "is_ambiguous"):
        continue
    n_tot, n_amb = con.execute(
        f'SELECT count(*), sum(CAST(is_ambiguous AS INT)) FROM "{t}"').fetchone()
    amb.append({"table": t, "rows": n_tot, "ambiguous_rows": int(n_amb or 0),
                "pct": (n_amb or 0) / n_tot * 100 if n_tot else 0})
amb = pd.DataFrame(amb)
if not amb.empty:
    print(amb.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    print(f"\ntotal ambiguous rows across all tables: {amb.ambiguous_rows.sum():,}")
else:
    print("no is_ambiguous columns found")

### 2.3 Roster membership

`cell_line_connection` is the identity hub — the union of every model_id seen in
any table. `in_roster` records whether `sample_info` knows about the line:
`sample_info` is authoritative but not complete, since `fusions` carries cell
lines whose ACH ids post-date its release.

In [ ]:
print(con.execute("""
    SELECT c.in_roster, count(*) AS n_lines, avg(m.n_modalities) AS mean_modalities
    FROM cell_line_connection c
    LEFT JOIN coverage_matrix m USING (model_id)
    GROUP BY c.in_roster ORDER BY c.in_roster DESC
""").df().to_string(index=False, float_format=lambda v: f"{v:.2f}"))

flags = ", ".join(f'sum(CAST(m."{l}" AS INT)) AS "{l}"' for l in MODALITY_LAYERS)
print("\nlines outside the roster, by layer holding them:")
print(con.execute(f"""
    SELECT {flags} FROM cell_line_connection c
    JOIN coverage_matrix m USING (model_id)
    WHERE NOT c.in_roster
""").df().T.rename(columns={0: "n_lines"}).query("n_lines > 0").to_string())

## 3. Completeness

### 3.1 Column missingness by table

Bucketed rather than per-column, so a 4-column table and a 54,000-column matrix
are equally readable. Columns at ≥95% missing are effectively empty.

In [ ]:
MISS_BINS   = [-0.001, 0.0, 10, 25, 50, 75, 95, 100.001]
MISS_LABELS = ["complete", "<10%", "10–25%", "25–50%", "50–75%", "75–95%", "≥95%"]
MISS_COLORS = ["#2E7D5B", "#7FB069", "#C9CE6E", "#EDC26B", "#E39A5C", "#D9704E", "#B33A3A"]


def missing_pct(con, table, chunk=400):
    """Missing % per column, computed inside DuckDB."""
    cs = cols_of(con, table)
    total = con.execute(f'SELECT count(*) FROM "{table}"').fetchone()[0]
    if total == 0 or not cs:
        return pd.Series(dtype=float)
    return chunked_agg(con, table, cs,
                       lambda c: f'100.0*sum(CASE WHEN "{c}" IS NULL THEN 1 ELSE 0 END)/{total}',
                       chunk)


def plot_missingness(con, tables, ncols=4):
    """Facet grid: share of each table's columns falling in each missingness band."""
    data = {}
    for t in tables:
        s = missing_pct(con, t)
        data[t] = None if s.empty else (pd.cut(s, bins=MISS_BINS, labels=MISS_LABELS)
                                          .value_counts().reindex(MISS_LABELS, fill_value=0))
    nrows = -(-len(tables) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 2.7 * nrows), squeeze=False)
    axes = axes.flatten()
    ypos = np.arange(len(MISS_LABELS))[::-1]

    for ax, t in zip(axes, tables):
        counts = data[t]
        if counts is None:
            ax.text(.5, .5, "no rows", ha="center", va="center", color="#999",
                    fontsize=9, transform=ax.transAxes)
            ax.set_title(t, fontsize=10, loc="left"); ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values(): sp.set_visible(False)
            continue
        n_total = int(counts.sum())
        ax.barh(ypos, counts.values, color=MISS_COLORS, height=.72, zorder=3)
        ax.set_yticks(ypos); ax.set_yticklabels(MISS_LABELS, fontsize=8)
        ax.set_title(f"{t}   ({n_total:,} cols)", fontsize=10, loc="left", pad=6)
        ax.set_xlim(0, max(counts.max() * 1.3, 1))
        style_axis(ax)
        for y, v in zip(ypos, counts.values):
            if v:
                ax.text(v + counts.max() * .03, y, f"{v:,} ({v/n_total*100:.0f}%)",
                        va="center", fontsize=7.5, color="#444")
    for ax in axes[len(tables):]:
        ax.axis("off")
    fig.legend(handles=[Patch(facecolor=c, label=l) for c, l in zip(MISS_COLORS, MISS_LABELS)],
               loc="lower center", ncol=7, frameon=False, fontsize=8.5, bbox_to_anchor=(.5, -.015))
    fig.suptitle("Column missingness distribution by table", fontsize=13, y=.995)
    fig.tight_layout(rect=[0, .03, 1, .97])
    return fig


plot_missingness(con, ALL_LAYERS);

### 3.2 Cell line coverage per layer

How many lines each layer reaches. The dashed line is the full roster, so the
shortfall per layer is visible directly.

In [ ]:
def plot_per_layer_coverage(con, layer_cols, table="coverage_matrix"):
    sums = ", ".join(f'sum(CAST("{c}" AS INT)) AS "{c}"' for c in layer_cols)
    row = con.execute(f'SELECT {sums} FROM "{table}"').df().iloc[0]
    total = con.execute(f'SELECT count(*) FROM "{table}"').fetchone()[0]

    r = (pd.DataFrame({"layer": row.index, "n_lines": row.values.astype(int)})
           .assign(pct=lambda d: d.n_lines / total * 100)
           .sort_values("n_lines", ascending=False).reset_index(drop=True))

    fig, ax = plt.subplots(figsize=(10, .46 * len(r) + 2))
    ypos = np.arange(len(r))[::-1]
    ax.barh(ypos, r.n_lines, color=[plt.get_cmap("RdYlGn")(p / 100) for p in r.pct],
            height=.72, edgecolor="white", lw=.8, zorder=3)
    ax.set_yticks(ypos); ax.set_yticklabels(r.layer, fontsize=10)
    ax.set_xlabel("cell lines with data", fontsize=10)
    ax.set_xlim(0, total * 1.18)
    ax.axvline(total, color="#999", ls="--", lw=1, zorder=2)
    ax.text(total, len(r) - .4, f"  roster = {total:,}", fontsize=8.5, color="#777", va="bottom")
    for y, n, p in zip(ypos, r.n_lines, r.pct):
        ax.text(n + total * .012, y, f"{n:,}   ({p:.1f}%)", va="center", fontsize=9, color="#333")
    ax.set_title(f"Cell line coverage per layer  —  {total:,} lines in roster",
                 fontsize=12.5, loc="left", pad=12)
    style_axis(ax)
    fig.tight_layout()
    return fig, r


_, per_layer = plot_per_layer_coverage(con, ALL_LAYERS)

### 3.3 Lines by number of modalities

Counted over the 9 **measurement** layers only. Annotation layers are excluded —
`sample_info` is ~100% by construction and would inflate every line's count.

In [ ]:
def plot_layer_counts(con, layer_cols, table="coverage_matrix"):
    n_max = len(layer_cols)
    expr = " + ".join(f'CAST("{c}" AS INT)' for c in layer_cols)
    r = con.execute(f'''WITH c AS (SELECT ({expr}) AS n FROM "{table}")
                        SELECT n AS n_layers, count(*) AS n_lines FROM c GROUP BY n''').df()
    r = (r.set_index("n_layers").reindex(range(n_max, -1, -1), fill_value=0).reset_index())
    total = int(r.n_lines.sum())
    r["pct"] = r.n_lines / total * 100

    fig, ax = plt.subplots(figsize=(11, 5.5))
    x = np.arange(len(r))
    ax.bar(x, r.n_lines, color=[plt.get_cmap("RdYlGn")(v / n_max) for v in r.n_layers],
           width=.74, edgecolor="white", lw=.8, zorder=3)
    ax.set_xticks(x); ax.set_xticklabels([int(v) for v in r.n_layers])
    ax.set_xlabel(f"number of modalities with data (out of {n_max})", fontsize=11, labelpad=8)
    ax.set_ylabel("number of cell lines", fontsize=11)
    ax.set_ylim(0, max(r.n_lines.max() * 1.18, 1))
    for xi, n, p in zip(x, r.n_lines, r.pct):
        if n:
            ax.text(xi, n + r.n_lines.max() * .015, f"{int(n):,}\n{p:.1f}%",
                    ha="center", va="bottom", fontsize=9, color="#333", linespacing=1.35)
    med = int(np.median(np.repeat(r.n_layers.values, r.n_lines.values.astype(int))))
    ax.set_title(f"Cell lines by modality count  —  {total:,} lines, median {med}",
                 fontsize=13, loc="left", pad=14)
    style_axis(ax, grid_axis="y")
    fig.tight_layout()
    return fig, r


_, modality_counts = plot_layer_counts(con, MODALITY_LAYERS)

### 3.4 Layer co-occurrence

**New.** Jaccard overlap between layers: which modalities travel together?
Layers that always co-occur (e.g. two CCLE assays run on the same panel) are
not independent evidence, which matters when weighting a confidence score.

In [ ]:
def layer_cooccurrence(con, layers, table="coverage_matrix"):
    cm = con.execute(f'SELECT * FROM "{table}"').df()
    L = [l for l in layers if l in cm.columns]
    M = cm[L].astype(bool)
    J = pd.DataFrame(index=L, columns=L, dtype=float)
    for a in L:
        for b in L:
            u = (M[a] | M[b]).sum()
            J.loc[a, b] = (M[a] & M[b]).sum() / u if u else 0.0
    return J.astype(float)


J = layer_cooccurrence(con, MODALITY_LAYERS)
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(J.values, cmap="YlGnBu", vmin=0, vmax=1)
ax.set_xticks(range(len(J))); ax.set_xticklabels(J.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(J))); ax.set_yticklabels(J.index, fontsize=9)
for i in range(len(J)):
    for j in range(len(J)):
        ax.text(j, i, f"{J.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if J.iloc[i, j] > .55 else "#333")
ax.set_title("Layer co-occurrence (Jaccard) — which modalities travel together",
             fontsize=12, loc="left", pad=12)
fig.colorbar(im, ax=ax, fraction=.046, pad=.04)
fig.tight_layout()

## 4. Who are the cell lines?

**New.** Nothing in the original notebook profiled the lines themselves — only
their data coverage. Lineage and disease mix determines what a ranking model
can generalise over.

In [ ]:
if has_col(con, "sample_info", "primary_disease"):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

    for ax, (col, title, n) in zip(axes, [
            ("lineage", "Top lineages", 15),
            ("primary_disease", "Top diseases", 15),
            ("sex", "Sex", 5)]):
        if not has_col(con, "sample_info", col):
            ax.axis("off"); continue
        d = con.execute(f'''SELECT coalesce("{col}", '(missing)') AS v, count(*) AS n
                            FROM sample_info GROUP BY 1 ORDER BY n DESC LIMIT {n}''').df()
        ypos = np.arange(len(d))[::-1]
        ax.barh(ypos, d.n, color="#4C72B0", height=.72, zorder=3)
        ax.set_yticks(ypos); ax.set_yticklabels(d.v, fontsize=8.5)
        for y, v in zip(ypos, d.n):
            ax.text(v, y, f" {v:,}", va="center", fontsize=8)
        ax.set_title(title, fontsize=11, loc="left", pad=8)
        ax.set_xlim(0, d.n.max() * 1.2)
        style_axis(ax)
    fig.suptitle("Cell line composition", fontsize=13, x=.01, ha="left")
    fig.tight_layout()

### 4.1 Does coverage depend on lineage?

If well-studied lineages are systematically better covered, a ranking that rewards
coverage will inherit that bias. Joins the hub to `sample_info` for lineage and to
`coverage_matrix` for the modality count.

In [ ]:
d = con.execute("""
    SELECT s.lineage, count(*) AS n_lines, avg(m.n_modalities) AS mean_modalities
    FROM cell_line_connection c
    JOIN sample_info s     USING (model_id)
    JOIN coverage_matrix m USING (model_id)
    WHERE s.lineage IS NOT NULL
    GROUP BY s.lineage HAVING count(*) >= 10
    ORDER BY mean_modalities DESC
""").df()

fig, ax = plt.subplots(figsize=(10, .34 * len(d) + 2))
ypos = np.arange(len(d))[::-1]
ax.barh(ypos, d.mean_modalities, color="#4C72B0", height=.7, zorder=3)
ax.set_yticks(ypos); ax.set_yticklabels(d.lineage, fontsize=8.5)
for y, m, n in zip(ypos, d.mean_modalities, d.n_lines):
    ax.text(m, y, f"  {m:.1f}  (n={n})", va="center", fontsize=8, color="#444")
ax.set_xlabel(f"mean modalities (of {len(MODALITY_LAYERS)})", fontsize=10)
ax.set_xlim(0, len(MODALITY_LAYERS) * 1.15)
ax.set_title("Mean modality coverage by lineage  (lineages with >=10 lines)",
             fontsize=12, loc="left", pad=10)
style_axis(ax)
fig.tight_layout()
print(f"spread: {d.mean_modalities.min():.1f} - {d.mean_modalities.max():.1f} modalities")

## 5. Scale and shape

The layers are on wildly different scales — log-ratios centred at 0, nTPM in
the hundreds. This is the evidence for per-layer standardisation before any
cross-layer comparison.

### 5.1 Value ranges per layer

In [ ]:
def layer_value_stats(con, table, max_cols=150, seed=7):
    """Quantiles of the numeric VALUES in a table, computed in DuckDB.
    Wide tables are sampled to `max_cols` columns and unpivoted."""
    cs = numeric_cols(con, table)
    if not cs:
        return None
    rng = np.random.default_rng(seed)
    if len(cs) > max_cols:
        cs = list(rng.choice(cs, max_cols, replace=False))
    sel = ", ".join(f'"{c}"' for c in cs)
    r = con.execute(f'''
        WITH u AS (SELECT CAST(value AS DOUBLE) AS v
                   FROM (UNPIVOT (SELECT {sel} FROM "{table}")
                         ON COLUMNS(*) INTO NAME k VALUE value)
                   WHERE value IS NOT NULL AND isfinite(CAST(value AS DOUBLE)))
        SELECT count(*) n, min(v) mn, max(v) mx, avg(v) mean, stddev_samp(v) sd,
               quantile_cont(v,0.01) p01, quantile_cont(v,0.25) q1,
               quantile_cont(v,0.50) med, quantile_cont(v,0.75) q3,
               quantile_cont(v,0.99) p99
        FROM u''').df().iloc[0].to_dict()
    if not r["n"]:
        return None
    r["table"] = table
    r["n_numeric_cols"] = len(numeric_cols(con, table))
    return r


def plot_scale_comparison(con, tables, max_cols=150):
    """Box plot per layer on a shared symlog axis — symlog because log-ratios
    are negative and nTPM reaches the hundreds; a linear axis flattens all but
    the largest. Whiskers are p1/p99; true min-max printed at right."""
    stats = [s for s in (layer_value_stats(con, t, max_cols) for t in tables) if s]
    df = pd.DataFrame(stats).sort_values("med", ascending=False).reset_index(drop=True)
    bxp = [dict(label=r.table, med=r.med, q1=r.q1, q3=r.q3,
                whislo=r.p01, whishi=r.p99, fliers=[]) for r in df.itertuples()]

    fig, ax = plt.subplots(figsize=(12, .52 * len(df) + 2.4))
    bp = ax.bxp(bxp, vert=False, showfliers=False, patch_artist=True, widths=.62)
    cmap = plt.get_cmap("viridis")
    for i, box in enumerate(bp["boxes"]):
        box.set(facecolor=cmap(i / max(len(df) - 1, 1)), alpha=.75, edgecolor="#444", lw=.8)
    for part in ("whiskers", "caps", "medians"):
        for a in bp[part]:
            a.set(color="#333", lw=1.1)
    ax.set_xscale("symlog", linthresh=1)
    ax.axvline(0, color="#BBB", lw=.9, zorder=0)
    ax.set_xlabel("value  (symlog — negatives, zero and large positives all visible)", fontsize=10.5)
    for i, r in enumerate(df.itertuples(), start=1):
        ax.text(1.005, (i - .5) / len(df), f"[{r.mn:.4g}, {r.mx:.4g}]",
                transform=ax.transAxes, va="center", fontsize=8, color="#666")
    ax.set_title("Value scale by layer — box = IQR, whiskers = p1–p99", fontsize=12.5,
                 loc="left", pad=12)
    style_axis(ax)
    fig.tight_layout()
    return fig, df


_, scale_stats = plot_scale_comparison(con, NUMERIC_LAYERS)
print(scale_stats[["table", "n_numeric_cols", "n", "mn", "q1", "med", "q3", "mx", "mean", "sd"]]
      .to_string(index=False, float_format=lambda v: f"{v:,.3g}"))

### 5.2 Per-column skewness

`|skew| > 1` suggests a log transform is needed before z-scoring, since
z-scores assume roughly symmetric spread.

In [ ]:
def column_skewness(con, table, chunk=300):
    """Skewness of every numeric column. Zero-variance columns return NaN
    and are dropped — a constant column has no shape to describe."""
    cs = numeric_cols(con, table)
    if not cs:
        return pd.Series(dtype=float)
    s = chunked_agg(con, table, cs, lambda c: f'skewness(CAST("{c}" AS DOUBLE))', chunk)
    return s.replace([np.inf, -np.inf], np.nan).dropna()


def plot_skewness_facets(con, tables, ncols=3, bins=60):
    data = {t: column_skewness(con, t) for t in tables}
    nrows = -(-len(tables) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.9 * ncols, 3.3 * nrows), squeeze=False)
    axes = axes.flatten()
    for ax, t in zip(axes, tables):
        s = data[t]
        if s.empty:
            ax.text(.5, .5, "no numeric columns", ha="center", va="center",
                    color="#999", fontsize=9, transform=ax.transAxes)
            ax.set_title(t, fontsize=10.5, loc="left"); ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values(): sp.set_visible(False)
            continue
        lo, hi = np.percentile(s, [0.5, 99.5])
        pad = max((hi - lo) * .12, .35)
        ax.axvspan(-.5, .5, color="#2E7D5B", alpha=.09, zorder=0)
        ax.hist(s.clip(lo - pad, hi + pad), bins=bins, color="#3C6E8F",
                edgecolor="white", lw=.35, zorder=3)
        ax.axvline(0, color="#888", lw=1, zorder=4)
        for v in (-1, 1):
            ax.axvline(v, color="#B33A3A", ls="--", lw=.95, zorder=4)
        ax.axvline(s.median(), color="#E08214", lw=1.6, zorder=5)
        ax.set_title(f"{t}   ({len(s):,} cols)", fontsize=10.5, loc="left", pad=6)
        ax.text(.985, .93, f"median {s.median():+.2f}\n{(s.abs() > 1).mean()*100:.0f}% |skew|>1",
                transform=ax.transAxes, ha="right", va="top", fontsize=8,
                color="#444", linespacing=1.5)
        ax.set_xlabel("skewness", fontsize=9); ax.set_ylabel("columns", fontsize=9)
        style_axis(ax, grid_axis="y")
    for ax in axes[len(tables):]:
        ax.axis("off")
    fig.suptitle("Per-column skewness  —  orange = median, red dashed = ±1, green = symmetric",
                 fontsize=12.5, x=.008, ha="left", y=.998)
    fig.tight_layout(rect=[0, 0, 1, .965])
    return fig, pd.concat([pd.DataFrame({"table": t, "column": s.index, "skewness": s.values})
                           for t, s in data.items() if not s.empty], ignore_index=True)


_, skew_long = plot_skewness_facets(con, NUMERIC_LAYERS)
print(skew_long.assign(needs_log=lambda d: d.skewness.abs() > 1)
      .groupby("table")["needs_log"].agg(["sum", "mean"])
      .rename(columns={"sum": "n_cols_skewed", "mean": "frac"}).to_string())

### 5.3 Fitted Gaussian per column

One bell curve per column, `N(mean, sd)`. Position shows where the column sits,
width shows its spread. Note these are *fitted* — for a skewed column the curve
looks fine while the real distribution isn't, so read alongside 5.2.

In [ ]:
def column_moments(con, table, chunk=250):
    cs = numeric_cols(con, table)
    if not cs:
        return pd.DataFrame(columns=["column", "mean", "sd"])
    mean = chunked_agg(con, table, cs, lambda c: f'avg(CAST("{c}" AS DOUBLE))', chunk)
    sd   = chunked_agg(con, table, cs, lambda c: f'stddev_samp(CAST("{c}" AS DOUBLE))', chunk)
    d = pd.DataFrame({"column": mean.index, "mean": mean.values, "sd": sd.values})
    return d.replace([np.inf, -np.inf], np.nan).dropna().query("sd > 0")


def plot_gaussian_per_column(con, tables, max_cols=250, ncols=3, grid=400, seed=7):
    nr = -(-len(tables) // ncols)
    fig, axes = plt.subplots(nr, ncols, figsize=(5.1 * ncols, 3.5 * nr), squeeze=False)
    axes = axes.flatten(); allm = []
    for ax, t in zip(axes, tables):
        m = column_moments(con, t)
        if m.empty:
            ax.axis("off"); continue
        m["table"] = t; allm.append(m)
        d = m.sample(max_cols, random_state=seed) if len(m) > max_cols else m
        lo = np.percentile(d["mean"] - 3 * d["sd"], 1)
        hi = np.percentile(d["mean"] + 3 * d["sd"], 99)
        xs = np.linspace(lo, hi, grid)
        cmap = plt.get_cmap("viridis")
        rank = d["mean"].rank(pct=True).to_numpy()
        for (mu, sd), r in zip(d[["mean", "sd"]].to_numpy(), rank):
            ax.plot(xs, np.exp(-.5 * ((xs - mu) / sd) ** 2) / (sd * np.sqrt(2 * np.pi)),
                    lw=.55, alpha=.32, color=cmap(r), zorder=3)
        ax.set_title(f"{t}   ({len(d):,} of {len(m):,} cols)", fontsize=10.5, loc="left", pad=6)
        ax.set_xlabel("value", fontsize=9); ax.set_ylabel("density", fontsize=9)
        style_axis(ax, grid_axis="both")
    for ax in axes[len(tables):]:
        ax.axis("off")
    fig.suptitle("Fitted Gaussian per column — N(column mean, column sd)",
                 fontsize=12.5, x=.008, ha="left", y=.998)
    fig.tight_layout(rect=[0, 0, 1, .965])
    return fig, pd.concat(allm, ignore_index=True) if allm else pd.DataFrame()


_, moments = plot_gaussian_per_column(con, NUMERIC_LAYERS)

## 6. Gene coverage

### 6.1 Gene sets per source

Genes are stored three ways: as column names (wide expression), as row values
(`hpa_rna`, `mutations`), and as two partner columns (`fusions`). Only real
ENSG ids are kept — `fusions` also carries viral reference accessions.

In [ ]:
def genes_from_long(con, table, gene_col, pattern=ENSG_PATTERN):
    if not has_col(con, table, gene_col):
        return set()
    return set(con.execute(f'''
        SELECT DISTINCT lower(trim("{gene_col}")) AS g
        FROM "{table}"
        WHERE "{gene_col}" IS NOT NULL
          AND regexp_matches(lower(trim("{gene_col}")), '{pattern}')
    ''').df()["g"])


def genes_from_wide(con, table, prefix="ensg"):
    return set(wide_genes(con, table, prefix))


def genes_from_two_cols(con, table, cols, pattern=ENSG_PATTERN):
    out = set()
    for col in cols:
        if has_col(con, table, col):
            out |= genes_from_long(con, table, col, pattern)
    return out


gene_sets = {
    "hpa_rna": genes_from_long(con, "hpa_rna", "gene"),
    "depmap_expr": genes_from_wide(con, "depmap_expr"),
    "geo_expr": genes_from_wide(con, "geo_expr"),
    "mutations": genes_from_long(con, "mutations", "ensemblgeneid"),
    "fusions": genes_from_two_cols(
        con, "fusions", ["gene1_ens_id", "gene2_ens_id", "gene1", "gene2"]
    ),
}
for n, s in gene_sets.items():
    print(f"{n:14s} {len(s):>7,} genes")
print(f"\nin ALL sources:  {len(set.intersection(*gene_sets.values())):,}")
print(f"in ANY source:   {len(set.union(*gene_sets.values())):,}")

### 6.2 Overlap between sources

A Venn caps out at 3 sets, so this is a pairwise heatmap plus exact-combination
counts — the same two questions an UpSet plot answers.

In [ ]:
def plot_gene_overlap_matrix(sets, top_n_combos=20):
    names = list(sets); n = len(names)
    M = np.array([[len(sets[a] & sets[b]) / len(sets[a]) * 100 if sets[a] else 0
                   for b in names] for a in names])
    all_genes = sorted(set.union(*sets.values()))
    membership = pd.DataFrame({k: [g in sets[k] for g in all_genes] for k in names}, index=all_genes)
    combos = membership.groupby(names).size().sort_values(ascending=False).head(top_n_combos)

    fig = plt.figure(figsize=(15, max(5.5, .32 * len(combos) + 2)))
    gs = fig.add_gridspec(1, 2, width_ratios=[.85, 1.5])
    axH, axC = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])

    im = axH.imshow(M, cmap="YlGnBu", vmin=0, vmax=100)
    axH.set_xticks(range(n)); axH.set_xticklabels(names, rotation=45, ha="right", fontsize=9)
    axH.set_yticks(range(n)); axH.set_yticklabels(names, fontsize=9)
    for i in range(n):
        for j in range(n):
            axH.text(j, i, f"{M[i, j]:.0f}", ha="center", va="center", fontsize=8.5,
                     color="white" if M[i, j] > 55 else "#333")
    axH.set_title("Row's genes also found in column (%)", fontsize=11, loc="left", pad=10)
    fig.colorbar(im, ax=axH, fraction=.046, pad=.04)

    labels = [" ∩ ".join(k for k, p in zip(names, c) if p) or "(none)" for c in combos.index]
    ypos = np.arange(len(combos))[::-1]
    axC.barh(ypos, combos.values, color="#4C72B0", height=.68)
    axC.set_yticks(ypos); axC.set_yticklabels(labels, fontsize=8.5)
    axC.set_xlabel("number of genes", fontsize=10)
    for y, v in zip(ypos, combos.values):
        axC.text(v, y, f"  {v:,}", va="center", fontsize=8.5)
    axC.set_title(f"Genes by exact source combination (top {len(combos)})",
                  fontsize=11, loc="left", pad=10)
    style_axis(axC)
    fig.subplots_adjust(left=.14, right=.96, top=.88, bottom=.12, wspace=.55)
    return fig, membership


_, gene_membership = plot_gene_overlap_matrix(gene_sets)

### 6.3 hpa_rna gene ↔ cell line pairs

Checks whether the pairing is clean — one row per (gene, model_id) — or whether
duplicates exist. Duplicates are expected here: the ambiguity explode credits
both candidate ACHs, producing two rows per affected pair.

In [ ]:
def hpa_rna_pair_counts(con, table="hpa_rna"):
    total = con.execute(f'SELECT count(*) FROM "{table}"').fetchone()[0]
    pairs = con.execute(f'SELECT count(*) FROM (SELECT DISTINCT gene, model_id FROM "{table}")').fetchone()[0]
    n_g = con.execute(f'SELECT count(DISTINCT gene) FROM "{table}"').fetchone()[0]
    n_l = con.execute(f'SELECT count(DISTINCT model_id) FROM "{table}"').fetchone()[0]
    dup = con.execute(f'''SELECT gene, model_id, count(*) AS n FROM "{table}"
                          GROUP BY 1,2 HAVING count(*) > 1 ORDER BY n DESC''').df()
    print(f"rows: {total:,} | distinct (gene, model_id) pairs: {pairs:,}")
    print(f"genes: {n_g:,} | cell lines: {n_l:,} | max possible pairs: {n_g*n_l:,}")
    print(f"matrix density: {pairs/(n_g*n_l)*100:.1f}%")
    print(f"duplicated pairs: {len(dup):,} ({dup.n.sum()-len(dup) if not dup.empty else 0:,} extra rows)")
    return pairs, dup


_, hpa_dups = hpa_rna_pair_counts(con)

### 6.4 Genes by share of cell lines covered

**Interpretation note.** The denominator matters: `depmap_expr` reaches ~1,412
lines, so no gene measured only by DepMap can exceed 1412/roster. Plotting
against the full roster produces an artificial ceiling, not a gene property.

In [ ]:
def gene_cell_line_counts(con, chunk=400, verbose=True):
    """Distinct cell lines per gene, per layer plus the union across layers."""
    per_layer, union_parts = {}, []
    for tbl in ("depmap_expr", "geo_expr"):
        gcols = wide_genes(con, tbl)
        if not gcols:
            continue
        parts = []
        for i in range(0, len(gcols), chunk):
            sel = ", ".join(f'"{c}"' for c in gcols[i:i + chunk])
            body = (f'(UNPIVOT (SELECT model_id, {sel} FROM "{tbl}") '
                    f'ON COLUMNS(* EXCLUDE (model_id)) INTO NAME gene_id VALUE v)')
            parts.append(con.execute(f'SELECT gene_id, count(DISTINCT model_id) AS n_lines '
                                     f'FROM {body} WHERE v IS NOT NULL GROUP BY gene_id').df())
            union_parts.append(f'SELECT gene_id, model_id FROM {body} WHERE v IS NOT NULL')
        per_layer[tbl] = pd.concat(parts, ignore_index=True)
        if verbose:
            print(f"  {tbl:14s} {len(gcols):>7,} gene columns")

    for tbl, col in (("hpa_rna", "gene"), ("mutations", "ensemblgeneid")):
        if not has_col(con, tbl, col):
            continue
        per_layer[tbl] = con.execute(f'''
            SELECT lower(trim("{col}")) AS gene_id, count(DISTINCT model_id) AS n_lines
            FROM "{tbl}" WHERE "{col}" IS NOT NULL AND model_id IS NOT NULL
              AND regexp_matches(lower(trim("{col}")), '{ENSG_PATTERN}')
            GROUP BY 1''').df()
        union_parts.append(f'''SELECT lower(trim("{col}")) AS gene_id, model_id FROM "{tbl}"
            WHERE "{col}" IS NOT NULL AND regexp_matches(lower(trim("{col}")), '{ENSG_PATTERN}')''')
        if verbose:
            print(f"  {tbl:14s} {len(per_layer[tbl]):>7,} genes")

    if has_col(con, "fusions", "gene1_ens_id"):
        fus = " UNION ALL ".join(
            f'''SELECT lower(trim("{c}")) AS gene_id, model_id FROM fusions
                WHERE "{c}" IS NOT NULL AND regexp_matches(lower(trim("{c}")), '{ENSG_PATTERN}')'''
            for c in ("gene1_ens_id", "gene2_ens_id"))
        per_layer["fusions"] = con.execute(
            f'SELECT gene_id, count(DISTINCT model_id) AS n_lines FROM ({fus}) GROUP BY 1').df()
        union_parts.append(fus)
        if verbose:
            print(f"  {'fusions':14s} {len(per_layer['fusions']):>7,} genes")

    out = None
    for name, d in per_layer.items():
        d = d.rename(columns={"n_lines": f"n_{name}"})
        out = d if out is None else out.merge(d, on="gene_id", how="outer")
    out = out.fillna(0)

    uni = con.execute(f'''SELECT gene_id, count(DISTINCT model_id) AS n_lines_any
        FROM ({" UNION ALL ".join(union_parts)}) WHERE model_id IS NOT NULL GROUP BY gene_id''').df()
    out = out.merge(uni, on="gene_id", how="outer").fillna(0)
    for c in out.columns:
        if c != "gene_id":
            out[c] = out[c].astype(int)
    return out.sort_values("n_lines_any", ascending=False).reset_index(drop=True)


gene_counts = gene_cell_line_counts(con)
print(f"\n{len(gene_counts):,} genes")

n_roster = con.execute("SELECT count(DISTINCT model_id) FROM cell_line_connection").fetchone()[0]
n_expr   = con.execute("SELECT count(DISTINCT model_id) FROM depmap_expr").fetchone()[0]
print(f"\nroster lines: {n_roster:,} | depmap_expr lines: {n_expr:,}")
print(f"max lines any gene reaches: {gene_counts.n_lines_any.max():,}")
for label, denom in [("vs full roster", n_roster), ("vs depmap lines", n_expr)]:
    pct = gene_counts.n_lines_any / denom * 100
    print(f"  {label:18s} >=90%: {(pct>=90).sum():>7,}   >=50%: {(pct>=50).sum():>7,}")

## 7. Do independent sources agree?

**New.** DepMap and GEO both measure RNA — same axis, so they *can* corroborate
or conflict. A low correlation would mean averaging them produces a number
matching neither, and the confidence score should treat disagreement explicitly.

In [ ]:
def source_agreement(con, a="depmap_expr", b="geo_expr", max_genes=300, min_lines=10, seed=7):
    """Per-gene Pearson correlation between two expression sources on the
    cell lines they share."""
    shared = sorted(set(wide_genes(con, a)) & set(wide_genes(con, b)))
    if not shared:
        return pd.DataFrame(), 0
    rng = np.random.default_rng(seed)
    if len(shared) > max_genes:
        shared = sorted(rng.choice(shared, max_genes, replace=False))
    sel = ", ".join(f'"{g}"' for g in shared)
    # GRAIN FIRST. Neither table is one row per model_id: depmap_expr is
    # profile-grain (a model can have RNA + WES + WGS profiles) and both tables
    # were exploded during harmonisation so an ambiguous line is credited to
    # every candidate ACH. Aligning on a duplicated index fans the join out and
    # the per-gene masks stop matching. Collapse to one value per line first —
    # the mean over a line's replicate profiles.
    da = (con.execute(f'SELECT model_id, {sel} FROM "{a}"').df()
            .groupby("model_id").mean(numeric_only=True))
    db = (con.execute(f'SELECT model_id, {sel} FROM "{b}"').df()
            .groupby("model_id").mean(numeric_only=True))
    lines = da.index.intersection(db.index)
    rows = []
    for g in shared:
        if g not in da.columns or g not in db.columns:
            continue
        x, y = da.loc[lines, g], db.loc[lines, g]
        m = x.notna() & y.notna()
        if m.sum() >= min_lines:
            rows.append({"gene": g, "n_lines": int(m.sum()), "r": x[m].corr(y[m])})
    return pd.DataFrame(rows), len(lines)


agree, n_shared_lines = source_agreement(con)
if not agree.empty:
    print(f"cell lines in both sources: {n_shared_lines:,}")
    print(f"genes tested: {len(agree):,} | median r = {agree.r.median():.3f}")
    print(f"  r > 0.8: {(agree.r > .8).mean()*100:5.1f}%")
    print(f"  r < 0.3: {(agree.r < .3).mean()*100:5.1f}%   <- genes where the sources disagree")

    fig, ax = plt.subplots(figsize=(9, 4.4))
    ax.hist(agree.r.dropna(), bins=50, color="#4C72B0", edgecolor="white", lw=.4, zorder=3)
    ax.axvline(agree.r.median(), color="#E08214", lw=2, zorder=4,
               label=f"median {agree.r.median():.2f}")
    ax.axvline(0, color="#888", lw=1, zorder=4)
    ax.set_xlabel("Pearson r  (depmap_expr vs geo_expr, per gene)", fontsize=10)
    ax.set_ylabel("genes", fontsize=10)
    ax.set_title(f"Same-axis agreement — {len(agree):,} genes over {n_shared_lines:,} shared lines",
                 fontsize=12, loc="left", pad=10)
    ax.legend(frameon=False, fontsize=9)
    style_axis(ax, grid_axis="y")
    fig.tight_layout()
else:
    print("no genes shared between the two expression sources")

## 8. Single-gene retrieval

The retrieval path used downstream: resolve a gene id or name, find every cell
line with evidence for it, and report what each layer contributes.

**Proteomics is reached differently from every other layer.** It is keyed on
UniProt accession, not ENSG, so a gene's `uniprot_ids` from the `gene` table is
the bridge. That is what lets `proteomics` keep its original headers — no
renaming, no averaging of isoforms, no duplicate columns. A gene with several
protein isoforms simply resolves to several columns, and a line counts as having
evidence if **any** of them is non-null.

In [ ]:
def resolve_gene(con, gene):
    """
    Returns (gene_id, gene_names, uniprot_ids). Accepts an ENSG id or ANY of
    the names in gene_names.

    gene_names is a LIST — one ENSG carries several names through aliasing and
    release drift — so lookup is by list membership rather than equality. An
    alias therefore resolves as well as the primary name.
    """
    g = str(gene).strip().lower()
    if g.startswith("ensg"):
        q = "SELECT gene_id, gene_names, uniprot_ids FROM gene WHERE gene_id = ?"
    else:
        q = ("SELECT gene_id, gene_names, uniprot_ids FROM gene "
             "WHERE list_contains(list_transform(gene_names, x -> lower(x)), ?)")
    row = con.execute(q, [g]).fetchone()
    if row is None:
        raise ValueError(f"'{gene}' not found in gene table (as id or name)")
    return row[0], list(row[1] or []), list(row[2] or [])


def proteomics_cols_for_gene(con, uniprot_ids, table="proteomics"):
    """Which proteomics columns belong to this gene. proteomics stays
    UniProt-keyed; gene.uniprot_ids is the bridge, so no header rename is
    needed. A gene with several isoforms returns several columns."""
    if not table_exists(con, table):
        return []
    have = {str(c).strip().lower() for c in cols_of(con, table)}
    return [u for u in uniprot_ids if u in have]


def model_ids_for_gene(con, gene_id, uniprot_ids=None):
    """Cell lines with ANY evidence for this gene, including proteomics."""
    ids = set()
    for tbl in ("depmap_expr", "geo_expr"):
        if has_col(con, tbl, gene_id):
            ids |= set(con.execute(f'SELECT DISTINCT model_id FROM "{tbl}" '
                                   f'WHERE "{gene_id}" IS NOT NULL').df()["model_id"].dropna())
    if has_col(con, "hpa_rna", "gene"):
        ids |= set(con.execute("SELECT DISTINCT model_id FROM hpa_rna WHERE lower(gene)=?",
                               [gene_id]).df()["model_id"].dropna())
    if has_col(con, "mutations", "ensemblgeneid"):
        ids |= set(con.execute("SELECT DISTINCT model_id FROM mutations "
                               "WHERE lower(ensemblgeneid)=?", [gene_id]).df()["model_id"].dropna())
    if has_col(con, "fusions", "gene1_ens_id"):
        ids |= set(con.execute("SELECT DISTINCT model_id FROM fusions "
                               "WHERE lower(gene1_ens_id)=? OR lower(gene2_ens_id)=?",
                               [gene_id, gene_id]).df()["model_id"].dropna())

    pcols = proteomics_cols_for_gene(con, uniprot_ids or [])
    if pcols:
        where = " OR ".join(f'"{c}" IS NOT NULL' for c in pcols)
        ids |= set(con.execute(f'SELECT DISTINCT model_id FROM proteomics '
                               f'WHERE {where}').df()["model_id"].dropna())
    return ids


def gene_evidence_summary(con, gene):
    """Per-layer evidence for one gene — how many lines each layer contributes.
    Proteomics is reached through gene.uniprot_ids rather than by gene id,
    since that table is keyed on UniProt accession."""
    gid, gnames, unis = resolve_gene(con, gene)
    per = {}

    for tbl in ("depmap_expr", "geo_expr"):
        if has_col(con, tbl, gid):
            per[tbl] = set(con.execute(f'SELECT DISTINCT model_id FROM "{tbl}" '
                                       f'WHERE "{gid}" IS NOT NULL').df()["model_id"].dropna())
    if has_col(con, "hpa_rna", "gene"):
        per["hpa_rna"] = set(con.execute("SELECT DISTINCT model_id FROM hpa_rna "
                                         "WHERE lower(gene)=?", [gid]).df()["model_id"].dropna())
    if has_col(con, "mutations", "ensemblgeneid"):
        per["mutations"] = set(con.execute("SELECT DISTINCT model_id FROM mutations "
                                           "WHERE lower(ensemblgeneid)=?", [gid]).df()["model_id"].dropna())
    if has_col(con, "fusions", "gene1_ens_id"):
        per["fusions"] = set(con.execute("SELECT DISTINCT model_id FROM fusions WHERE "
                                         "lower(gene1_ens_id)=? OR lower(gene2_ens_id)=?",
                                         [gid, gid]).df()["model_id"].dropna())

    pcols = proteomics_cols_for_gene(con, unis)
    if pcols:
        where = " OR ".join(f'"{c}" IS NOT NULL' for c in pcols)
        per["proteomics"] = set(con.execute(f'SELECT DISTINCT model_id FROM proteomics '
                                            f'WHERE {where}').df()["model_id"].dropna())

    union = set().union(*per.values()) if per else set()
    print(f"{gene} -> {gid}  names: {gnames}")
    print(f"  uniprot_ids: {unis or '(none)'}  -> found in proteomics: {pcols or '(none)'}")
    for k, v in per.items():
        print(f"  {k:14s} {len(v):>6,} lines")
    print(f"  {'UNION':14s} {len(union):>6,} lines")
    return per, union, unis


PROBE_GENE = "itga3"
per_layer_ev, union_lines, uniprots = gene_evidence_summary(con, PROBE_GENE)

### 8.1 Isoform breakdown for one gene

A gene with several UniProt isoforms has one proteomics column each, and each can
be null independently. This shows how many isoforms were actually measured per
cell line — relevant because averaging across isoforms treats a line with one
measurement the same as a line with five.

In [ ]:
gid, gname, unis = resolve_gene(con, PROBE_GENE)
pcols = proteomics_cols_for_gene(con, unis)
sel = ", ".join(f'"{c}"' for c in pcols)
df = con.execute(f"SELECT model_id, {sel} FROM proteomics").df()

print(f"total lines: {len(df)}")
print("\nnon-null per isoform:")
print(df[pcols].notna().sum().to_string())
print("\nisoforms measured per line:")
print(df[pcols].notna().sum(axis=1).value_counts().sort_index().to_string())

### 8.2 Validate the retrieval

Recomputes the cell line set independently and spot-checks against the raw
tables. A validator that can only ever pass is not testing anything, so this
compares set membership rather than just counting.

In [ ]:
def validate_gene_fetch(con, gene, model_ids):
    """
    Recomputes the cell line set independently and compares. Both calls must
    include uniprot_ids, or proteomics is skipped on one side and the sets
    disagree for the wrong reason.
    """
    gid, _, unis = resolve_gene(con, gene)
    expected = model_ids_for_gene(con, gid, unis)
    ok = expected == set(model_ids)
    print(f"[1] model_ids: expected {len(expected):,}, got {len(set(model_ids)):,}  "
          f"{'PASS' if ok else 'FAIL'}")
    if not ok:
        print(f"    missing: {sorted(expected - set(model_ids))[:5]}")
        print(f"    extra:   {sorted(set(model_ids) - expected)[:5]}")

    probe = sorted(expected)[0] if expected else None
    if probe and has_col(con, "mutations", "ensemblgeneid"):
        raw = con.execute("SELECT count(*) FROM mutations WHERE model_id=? "
                          "AND lower(ensemblgeneid)=?", [probe, gid]).fetchone()[0]
        print(f"[2] spot-check {probe}: {raw} mutation rows for this gene")

    pcols = proteomics_cols_for_gene(con, unis)
    if pcols:
        where = " OR ".join(f'"{c}" IS NOT NULL' for c in pcols)
        n = con.execute(f"SELECT count(DISTINCT model_id) FROM proteomics "
                        f"WHERE {where}").fetchone()[0]
        print(f"[3] proteomics via {len(pcols)} uniprot col(s): {n:,} lines")

    return ok


validate_gene_fetch(con, PROBE_GENE, union_lines)

## 9. Summary

Key numbers pulled together for the write-up.

In [ ]:
print("=" * 62)
print("DATABASE SUMMARY")
print("=" * 62)
print(f"tables:                 {len(summary):,}")
print(f"total rows:             {summary.rows.sum():,}")
print(f"cell lines (roster):    {n_roster:,}")
if table_exists(con, "cell_line_connection"):
    n_all = con.execute("SELECT count(*) FROM cell_line_connection").fetchone()[0]
    n_out = con.execute("SELECT count(*) FROM cell_line_connection WHERE NOT in_roster").fetchone()[0]
    print(f"cell lines (any table): {n_all:,}  ({n_out:,} outside the roster)")
if table_exists(con, "gene"):
    n_gene = con.execute("SELECT count(*) FROM gene").fetchone()[0]
    n_named = con.execute("SELECT count(*) FROM gene WHERE n_gene_names > 0").fetchone()[0]
    n_multi = con.execute("SELECT count(*) FROM gene WHERE n_gene_names > 1").fetchone()[0]
    print(f"genes:                  {n_gene:,}  ({n_named:,} named, {n_multi:,} with aliases)")
if not amb.empty:
    print(f"ambiguous rows:         {amb.ambiguous_rows.sum():,}")
if not agree.empty:
    print(f"depmap-geo median r:    {agree.r.median():.3f}")
print(f"\nmodality coverage: median {int(np.median(np.repeat(modality_counts.n_layers.values, modality_counts.n_lines.values.astype(int))))} of {len(MODALITY_LAYERS)}")

print("\nconnection closed")